# Persona Continuity — train + run in Google Colab (A100 tuned)

Trains a LoRA on the synthetic corpus already sitting in your Google Drive, builds a RAG index, then lets you ask questions interactively.

**Recommended runtime: A100 40GB** (Colab Pro / Pro+).
Defaults below are tuned for A100: **Qwen2.5-7B 4-bit, MAX_SEQ=4096, batch=4, grad-accum=4** (effective batch 16). Expect **~25–40 min per persona**.

Other GPUs:
- **L4 24GB (Pro):** keep 7B but drop `MAX_SEQ` to 2048 and `BATCH_SIZE` to 2.
- **T4 16GB (Free):** flip `MODEL_SIZE = '3b'`, `MAX_SEQ = 2048`, `BATCH_SIZE = 2`.

**Workflow:** Runtime → Change runtime type → GPU → A100. Then Run All.

## 1. Check GPU

In [ ]:
!nvidia-smi

## 2. Mount Google Drive (where the corpus lives)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_ROOT = '/content/drive/MyDrive/Projects/ProjectRecall'
assert os.path.isdir(PROJECT_ROOT), f'Not found: {PROJECT_ROOT} — adjust path'
os.chdir(PROJECT_ROOT)
print('Working in:', os.getcwd())
!ls

In [ ]:
# Bootstrap: write personas/cast/accounts/projects JSON directly,
# in case Google Drive sync hasn't uploaded them yet.
import os, json, gzip, base64
_BLOB = "H4sIALfR8mkC/719C3PbRrbmX+mbOxNJFYImKcmWldp7R5b80MSyZFGJZ2Y9pWoCTRIhCDB4SObM5v72Pd/pbqABEhRpZ7cq5Yh4dp8+j+88+uDf3y1UmiWxzJ4t0nApu7/Sj+9OxXf//hwL8fm7MPhMvz5/xyfvs6lM5/Lzdx19clxE0X0s50pfc4NrxLB+jZrLMHKe0dXP+EucpPn0MYwDL5My6/rJvLwlTSLzxKGKwyQV50WWJ3OVimHh+yrLxJWM5USl5R1090LGS9zE48axalwf7KvEUMqhuUnwgJKg8PPmVedRUgTiczHo9Y+EFK8Gr8Rjks7GUfIoZEEjkXmYxGIRyXycpHOxn0/DeCb+IRchjfEHMVTpQ+irD3T5KFJxoAJB14l5GHhzmc5ULpJFJnIl59mBM5qAHhzGejCt1MGF09/0RWdEljDuiLu/OSez8F9m2v2TnlDzRZQslco6GFWoMvGqI/7nT4OTK3F2e/v5O9z1uyFiruIiVTUa/pqEsTIcMOgNBl7vudc/dMesFjLN6bZ7GcdJEfvO1cde76XXP3aujmSW3wdy6VzSH3j9l+7w9QonqZli+msRiys1zWV9sHPLAbjqvYqluCHSiv1fbla45aBiLLUguma46X//0xzDkXEShUlt4tL3aTb5Pf+LM0e9cog5rfL9PPyi330iVJyrlHg7Ux0xGDjL3BH9nhhevXKml6oJsY4egWE5cUZjDX0p9ukZdOMS46VLX1+9PhP7/YEwY8E8/lk+iFbvPlXZgh4WjsIozA1R/3TUHVw11jWJ1T2x+jiM6quLwWS5JeIjiaWHy4hkNEYZOcMGm8vqJXMVhMXcSog+KR5DmkusHoWfxLn080xIkiX1RfnEfL7MCnuNynI5isJsSnJRXTumgYh8mioZiMMfnHc/qHSUZC3vHpEGUnQ75Isemxf0LppDpvhIqiIW1WwaLrxREUZBGE9cQVLBhI/gwZBu89QgTJWfC5nN9CTs7zh7JGXpPqCYW05NfJoj060jgnTZEUkcLfWEQzAInWCRd+5WX/xIamVyX2RGalOZKnMzpjAhoSQZpAfFmdjfO/PniqZFZFbBf+y52kPNk1/D6jEZySVNLbIDKccwjKQ/87J8GSnBupke+7kITg4D+tc/Coj18Munf8cnL80vRf8GfXXQoQWm9aBV1U+rM1oWTmLJymAhc1yQ1djNDuF+kiqVG7qTFLxTS/FvaOvfNeO/C83Pjv59laQxXW0OmjVyZcGOpnzw/ThMSdPkSeFP7UvKh4ppslBimRSpeFRqJsJMJOOxyBNS9ZMkCcBHad7V775QMq1Gs+mVDlevvLIxL8tmvxWhP6PlUafuo0sygZw0MvO0u6mMZ9l/fP4cm7tv9FPN//RBtsD6wPlUEa926HpzcM3g6294RVNwrsdD9EvLg9a2f46tYR5eif8jmvbVeRMY7L7ii1GU+LM1YKF6YJul/xzX3yLovWKNlfwc/9AX+8f9wYE4PiYL1D8a8KXnEpY4Wp6SKopwpYY7FtC4bPyQ+HJELBRPWK1r/V9yMWnRiPT/YprKTGX303AyvR+n6rdCxf5SU5KvZFgSpn4E1h1JTLtTnSmXvnaUeHAcyUnt2CPbCIlhR7UTU7lYLMG2jzKaQXMmxWRauwL6Q5JaJ3WRZbUzULvEBaxh8qkyerd2SRjzmYwYVowSmddOks4VdHwWk858nEr6IWeELTIVZ/UZxYlIi2yqCSxExRkNMsqHJAys5JR3EzQV86UAbBAGRzrPlhkp5swvskzVhz4LsdK1Q4tI0XvESAkZPITNG15fX6yO0CfWINNEtplQlk9IIM7ylNBiZb/L++8eEwEMOMlKmwgFJLrdLtGE7gzwZ+2VH5kBzDqcrpy+JpVP5mf1xBviD9is1TOfpipV4pKoBcRpzq+QnQidquA+T8mYhc5cIOo5rXYYGG11rdc/oX9SQcDWV+bEEJBP5opMS6mfibJpIgM+9E9XlgLlh7CK92xwaoIUqLEsopzOSIKNWinQ4pDw5apkTN/qA6sVYdBWrjI4vjRLzuKTDspIdLSRBZtn0yQyGNWcIcYlKWIMuW9g5QFxtYoBkcV/iT8d99hKEH2IF2bi+pbXDcxBmOZZpCZkVsMsK5QLDRbE9hD7auZkIJNxbjkEliDTc2TOIuhPgDHHa0klk+omuCFkxBPKwwdFxv+SlEC8lxMmyGG/xMc+zTz0oWCgVxXZ7lGRC75MYJqGhClpABo7KGbUhHijRqJ/LMIx2UAB/SxGS355DVNkcgnTFidmdaJHuczEQoYYHWlfDW+kWKqS8ffoeXqUQSL+pgekD+HA3/ecp5MSnod5Rmb6Pg/nihSlyvSLlvBX+NkAbblHumhUgCg/ioTgh4c7izj0ee1oDnZlkrHIonBRV+cGJat7LLXR0auYZFRkeD8pdAIGZhgFCVIqDnsE6WNn3OU9jFndG2Lo1LmGKx0BJEfIKyYLkddQn7md1i5JCdfeG4DvvnQgzGPXGG2Sl0Cmhoe1fjaDF+xblbcA3pDRux+pqXwILVD1p8onXDukF8mlIDhnRqsWEfxDRoskDxZ4KpI++tsPa9ydFPl9Mqb/xuToWrhn/AiaE3M2GC0fF5G4vr423gENltC0gsjkqumlLEI/u08e45oZwPoR2pUEU78n+V5AbSXuanx8ReIoAxqhYqzOKjWuE1z5BUTIY0Zo+AXurPxpkcaalfI0JEXgzjgmK5jCgRAEigJAxnGazAmMRDXKjBVDHY8hAdkArax84wrc2IhDeX2pzEiJTVI5F/s+LFWWF0EIMUgV8T1mV4sVkMvHwOL7UhWJSs9lmrLWwy35tTFtRwwcB9powI6AJz080GtKhi8jLytQxCSkgOf0zimxSTYjvyoiFiKfZwI3DPqmiB8kSQC7d3RfPnWGfSVT0ujiVtHwl2L/7HWHTA7RT7AAENwi5G1eaTBVTqeiSI6SVAdd6L+SC8i1KZ0uMC3QBy6w6/NbYXzW8v3vZBzLqTiHlt0fJlHB5BCvYzKpBJfNqyeJByQG46L8KfRMRFOfJ/BjI/iWiAaB+i5BnbdchGqSiLOI6KD+JfbNmlsoazXlJU14omeVHdTNX1bZPcNOwrBT9qPI6Dj0JTm9WBWo9QDaGwZ4LhcQL9LSznDuCBYXvZ7qg/BQ1fvDYoGwB9k8GRw0beFQPXiDZ4dw2VmMtcGYhvMOvTBUNWhVC9CI/TJ4Yx5K6DpIHkEp0k8kn9dEBUR9yENOcJSGGo4UGyazZs6TP8gglOI6oufRYt2qh+uFpdOCvP7MxkVI6eWSnqh4lUh0CRenyQNpjLqCMXHO6J6wdzrLajrGWLbKJrMmJ9HyU5VDGNUCRgZniACwM2TP6oiUNDfotVcHdntQpaQn4tyLaH3YEZgQRWMdEkS4BDxPRpx4mgSILJ1sovFSpc61qymyYo44kXXfoVjovpdspwpcSOPDUOfWza7CT9ok8Uh16EQsaCX0i62Fsc+lkZP3onB2f6wUHyGcC+QoGQbL6ADRFsVohXAVcRSgITH1Ypo1CWMjNXrFIw5JVFEJ/cpaaFY/1vptdB95omz+ydmIEUzy+RT5536SQDGzKLmaFT5FJowVhHMS1CZl5DAjgxgUkUaBVwlbxRPSwzq6WlelxkgRYuGH13gIMWiid45lZT4yzzcTyeEiqLlIRlFIogz1t1cBbKJstNcReyQmxXxEw6vhsJJMi4QUGo0AtANTgBvsNOSD0nEwBMxq0btsQQZQaypBmpQQmX6pDSzaR5DYkMLLoY3jMbljcR4aBco0+Bz//jkmxF9lDNKExLYtY8An78OlE6Zv5AtucYW4dK9wsgX64bh/y1xBLknnDwliw+UtdXpH3Jg4/bdmC7aOzxNCA+7XUmqiHUvCQzHxIGcGgLLETwUJQkzqhczKTZLlEwKqwoSAO6SpybftiPOItMm7pDABTaC1nA5/OLsb6pwCqVs5YQB3q4ACWNcQp3fEnUpTyfkJetnZp+Guwf6e1zvxekdbB/v7dEP/yWD/Yb892H8ll1J8ArKR8wzh8GSewLiR6NyRIbbmqjUNcKEewpkUtzIR+xccsiUKkS60rMBkstxQZQV0VJZXyWZ0cMhKxQtS+fp+IjzB00yvAvnEC/ZZOsgJTTvi7OZSANaSJek44daLn7LqTdtF4nW42QPwoeObAvGIWRN16jFxeg4wCuDmQo96Dg9dRy9o9QJoD7KU8PqCxNchbj9NsszjeWvzk7WH4PFWMJ8/lfkzlljrPXQE4mF8svmCBbE4jSNX86wtAt9IK0RykglwWZrDOV4C+JE4hDnrzjjJRb/X+zNZQ4Qfihzq85LDMDP8OSkQzlhmNUXqxOs5Rl+CTNcW0bTimMydDXY7pvGZxWXeWLLfbZTwU0F9GRF6zPXz2mP2OjQPB+xUOCH5ntiPkgSA86AWjj/qVFcpOSD4RTBUhDlfNXjRO3bOB4NjsU/GhbX6wU5hex3DbcTsTURb2RhQFcM3B94kEQLHa2PBVchZL3Wqb8k52OyGswVbCHPEeRQx1T2ELkkx5fvREnJoc6I4fCoq28LhX4KPoULyph43g+ThxD0iCkSZ+2RB3mZqY2JxmJ/qtyNGQ7RLzU+OYyMKp3+OH8PHMi+BAz/0SSwbobDNYeWNcWQ8vx7ZzVPCZCb6RcqC8M/BmuAwrWc9IjziUGpKyLqoB4RppvUAL4FVb6Q4h1QPUZOvSeyv0Vz9DGJNU0UeCDE5nDIR1t/BgeIgQZzYBPfB1gTsZvWY8aPai6BRiOUAuUNGO1n4RbudWTMQvUe4nXQBAUYvWZBjxPnvbwnmvv/x4nY1vIoYL55PN5yKfXnAwd390cHKhZd7pOyKbMpYH7aXpzBSvoQVX43aSoydUCFhQ+CWRyV0YBH+dRI90AL898odeyZyCHIjyMDBZ/pXP+uBnRh6KfzU7L9XqWGi7Tb6brj9V9Jtmn0JyxdRgGi5ktnSHAvni8j8HfIA0vCBjdO3BHyzmSKawsEmOw1vKIzHKXlzo5AGBstPWI4GkqQ8PwIBHYGQQgTu0VA280lmN8QskcPde6xolpH9jEAzy6XsLIWZph7xHnkKpFT+e4+mP07Y8x4jvDapASGosXujO9ZFv27fnHfY5DHz/qctFfHSsd/h0CxBwIUO9RIe1ME+E/2V4rDnkScn9OOtF2fO4mBWCzhZrd4M85HNQSY6fDAurY2xdkQZoVOOVYaHn6loLNhSGQ8S97jxMWJQ1D7Q27TqNTT2EV0jfjUnxK8FknmlXwFcR/gonhXpDqFZQ2QACGOxFXnhmiil0zs4mtYiyoEyq2JWQ191VEVIm2HVDDb3voQ8DEOMF80pAD23jvaYHon0wg3yut5mMxgMX7vhWSGMT5SNwjH9v8hhUVhGOwQwfBIEERTM6HZJa7FJgzzuNfK4t0GC05KOAdwHcm2Aj3jsNqYTuMNPkyJ34jrnw6st47B11Cv2Kz/pNXsltuyJvJPXQxd0wVLDl/S1B/q9GA6v3QhShZsZKHI4KCI93gBWNUgt9rVHRW7OcqGGfhoucvedpUMF734uhQ2a10IMFu0nsYfsKpEmb8YQVOU6kICkOtWSLMT+cJlBaC6YSQ92Dri6rorNOem1ZADN8daOiamQsYKu0k8ibwNQjD1Ik0BA1J7zybXAXNOZUjry2H85EM0wXZTEEw+i7sRak7RjnVEWawW9QU8j3iROCvPSj6pcMzfkKNOCPFy5SAij6HcPP71uRnaNiwCnMoajXkVzBfLyC6R3ebZz5i/4z9qoVLwDnSJeISa7r5WO8x47bPbZKq6isTwMNkRrr2yQMZUT4Hi/FubtsG1HPMtY95srbYLIgyL95Dy3UZRmQ+nOJJ2Ivc4llFFfaAbnUUPUKYifwjnxnfILJI3c2K0OSq/EySF3z0jWWKqsShFW524fHK1MGhszEhv6FxVuAVHWOklZRI93pcuEVCv7q6EfA1YaKZOvxH9kbSuf1Kh4122TyDfuTVL5oJYkxXtwK6FJTXSc1LoSM1QUBDzZx+myBvHHHIFkRzmdcIrgTRoGGF4RM5RgQCPjikpjlPfVs+ao2CTNryOQ2iaXFg4kWJbxWZv64ZQIjRvJTwMC6ScNnabLBQIN02UtPnP5qfbjiX3JsAD/xpNIeWQtAnJ+o0k+h5MbE+Ph/5myNRx7B9tHLBcqWdAztf6BtIhJkmW1VMaa0GE6YQ95otwo5EqIME1+RShRhwjv7c9mrJBgWu4GA2kVYHgYW+jKOwMnxThJcvaa9AIciyU5J1lXvJakfczjBfikvIOhD1DVTC1tTKOjQxE8bxr9A+FrVVb6Fhpx6htpLWTEzoKOTT8zi/uMxAC5UaXlrFvGduwUHQr/26mmsSUHbCo9EnOkCMd+3YMKcxvMrJtUTnHwPYIgmmK3xY66tHQPGaLXk+TiFf6sAoe1V4BqZUDu0Pt4WDurKygt2oIoBM3XmLy8E5ckmo8LTnl+POS8BzxcBnPOk4Hd7jGD0lUnE6fcV7vR0iGwEAYIpdM1vrxMJwXke0Lqh4TLznXfamdSqCR2XONMi0gjS7NnWTHKGBto08n3iHlo0m6kuImHoA5MOvWg607M+vxIrxVQnFhfjAA4i5iC83IT4pXYRASJnYq56oo3Ycy5KMOIJCU8IRSTp/40BBLUKT1Ie5djExlZVHKoJoQDTsVx78pEep+Rkqr7i5p3jcdWC6A3bb6p4HMssS1ptNDD/HbtXxVlYR2e5uFYlkz9Ha3GqUV7mh0Nd9DKY7n0A88ubr1er/+i/ixLDhNpNexUIzixkHqQUaEXZ8KJTfrjv1booZ/6e6dVzGDtPWPtPcfaN6Ts1gUFqWJDJ/YfELyZqdgbFT6K9Q1qehiISMnZ0hw+aJero1ahgl5b0Kxwkdc7+QYRoaXmp8CPIO2N2GgC/YXwDDg34OIu8B5Nh5MJ9UnpMpNRgTK1PJVwQ62gMcAJSoCDpI0lTkfo7QzFQqsBlyCCGCiUcd4VDioLwkA7lIgesI3Ri6tfb3W+lYSfYtL+ekH4slNS1alH5o8eqw/rWE+2RO6BfDMErYWpqRf7iA+Q2I1yjh1PdFocMnGwgwxVg2+VqSsVivfIBD0pK1wnh3FrJjSMrx3uU8MFxx7XsdNCeGYhzJqaG27rjxH/eTTobxKt9+6agOxggtoTsA2CBI8j/pp4TFEi3VilpO62kTCAS498CC+T88gjZNFqyhiBunrP6Izh2dV78YO4vrw4J4+ECxK+WaQ+Hnb0A9+edfQLRiqX3yBmth7ZY1AQi/13SRKFHfHzfER2J5Id8TZKRuoLWY5XKFXjt3u6+tOG/Ain+iQoMVAywSmGIFE4SuFamRKVH/VgZcDCxbKR0ZslEnLO/hJUodDdXfF3QgHwLlSeJg+ln8bQ6UcjxVqmaDGF9h5Kz+FAINGXc9Wfl4zHO8hG7bWmGtS6Jl9nTMAc5LeNQ4izwySuJRkcnjpk3cT61VWWoIRNpHY2uuKvl3ccHeUbgNNLC1P6XcPzyyvLjFtJwSL0TOzCK2FFmyC4gY4Kgxhp+CnRqXxYmjD2pmwQXxNCJVGWzXrlrxGN/kBXmsjoR/0y6PUlsUkepvqaY+/j4BtEBQ/dQ32Hr2J4K6T3AxWRhC+iYgK/kd3GJA8fmOxzwvEAajxHbU04Vvc20SVtSQRPM1bWKj1KaPqo8HPJ+boQU6EJVBE2bQCJAAgAc4Yd0+Qn6wU2Iym9NngSaRFpO6H8JNMxHZQHROoLiUpXXCCa7qwWDfBQLKaImgO4ZUWU63zqYe/PwkQxaZiBTi2gZmjx8uUfjeC2tz8uy62zQDeYiTcg+uSJTnUa/1ezTG+TsDkLt7pmazhstLRMto1k2ZF4lZFsk6tLc+k2BnVnGSKMyk5ixw2VL4pyp89XiwsbEgJZuT/1fk1GYhaigpu3Iv3P4RdsbcGmOhtK0K4F+Kl/dDoYiJ/vyC8iZFcLazWwH6q7LAFMuLfDPP84VdEcMqIMlNdeZlccvShLyALUJgaYd39QHmQoqGnZFXfaXdEb7ep42zrkWzF+lZeG5MG+cSy0xujr5cCFaptEYT3m2sTbZ1p8Q1QrnDZn13FRqYxUqtMzZWmAobln8kIW3e7I9McY5WCwFdOba1ncq8GighOV0YKGg5gBLTgzXbsYHP//F4OrMONqswkXpjThPrQ4j5mdBDIsebT0uP6XgVIGz4fcZrZyEwJiqN4IU7+gOwmfyZlKyRvh2dMFL/8MKAYaWakA6x6eWPa2luYHAb4riwpV8KPQm9V0oCyo4AL4lW4bvj9D9JfETm8DCeqmgSAo723leXkhEVLbEmR+djENrjA0ttp/vTBUbAYwV01krubJJgm506tUzYaGQvwIYlmRwOJpn4MkOkSgzegbhrnbyMPCRAm8OREzNL5Km0SUgTD3YsH7ibk6iaRhgwHwPj6Jo57gcgKaWSubnzGyrw2G64oK75HUu9dnp6DI9K9BVzMcMT1vAi/rNPev4PKC0gciYGSCsPK3wYsKTGyhnnRey8uCGWKYOqUW/msj6r100naicY/N4pHI3Q0PNgUqt8G537I+PxsfBOkJuLY0IsIt5FP9QHKLwAX+tgl2G3goM5K2rJOcwHRJBL+eJfHsMbGOWcMlS3dYsNrztlmjzA95F3/p5bS65XB1Bt1e3SV60hEn/Nb/f7kQ2ouWebJg35B1JKFwdr6B0itfuNwXtK9d8I62FAfbKBYThvcRe2RXy8sWzTIkh1ZOjJKvc2px7dpvpNignWL8OO1xcKLexmC+waRqoJQS3xx6vMOdX9IVv2gMiQDaUuW8n8FAuHJfpNg/Or5CkBUpBIRdzRUHtAZG/Z/qIdKBxC84aWRIEsYBp8T0voYvOWdoNrD6Kp5zmf9pTtf5Po93G/wWWTe81TpY//Atrv/4XoMXv/SO9n+6fP/+9cXBRhPRzvl6MN+waDdXqFlWk+SgcmU5pguptGO2g632uTK1TQKd9ZHdUWTBjMmKkkXXHTaQvjJpRltj+Xp4RzYbtmkpdFajU75RR6Q42Pfh2T/YSn34oa+3yHH9uUwnnO0YkTSOw7xLtuRBZwZtcLeRtt1B+TWCSc3kxZPe7/ql1g96awcFlZIuT5s03oED5yHMgVYDpEciAjwmn/QkM/LVQKWvzH4+8xgsZSKOhPvobAsePdwEY7bi0Y3K+ULGoYrEOyXzqRNxwSQY8JlZrEkknPLGYDKfyL4rrq/voNQxSLBVOFbcIonwa2lJ98Q5KcU0mZOqPwXDc/lmoLenXsYPCahU5WEFVwjAWrB5fXLpdC2PpyM+ni488FDZ05qQXS0EsnFstxJIVweR/uZfXxNwKBMl37ZUegW4iqscDu+PdmesEQwrg7QYpUgBfSj/5r1oumaN07wa86BOqiycIkKobCsgb+sfq120HumCVvmoKrPKTbejJMl9OV+I5n0NJb3RjY24lg/VGigrir+dwNgRhDC9LmzZ04H/w57g4hJUB5mZ7IlJQf56Rat6uYRF+Lopid0QvqlcwjoIZfsqLO5Rr/QVTGUEKiJUzJV+yD0QyPVRPPRUlYNb4SDe6CKbwPFDuL9VIOok+1+wbwfcnYADRbFMU926QKZ+9qOuQcz4PLoTZSj5IsYosnAUKc1qiLmWFRX2dZsrKqQ/b7QcKXdycbOk8yRtCrXdZ1N16mp0Pgmwr3Zp/Rf+FQLmEzjmPfLPuCdcEOL4qFjxfmx/tnPeGi2xGev6Xe0K7r12Kk6Oer2ey2BhWRjNyrzXd/un6SoT9ijuTd8sXVPoTHkoUzmtwgSOTP1yI64XZk2N6+dststwX5dW8S+gp080M3vanNqwcv+P3QbcERm2ApnqUywnFHkm4iKYoBbsd2fg6IJCyty/HxVLTf/auK8AIXl38srAz99crwx3DkCb4vJtxhtjS2WamRxeR4wi7rbgDo/LRapub/2jQf3kve8Uyt9ccxUt5KRqB2iq5DWLpK7msr0DzdTObky1b2i2qmuk4KIZ49rcBzJXbju9oyY7PI0x7bn6dsSPfY792LoXsPKiyJWurcpI3kW1MxQZKVoDYfYwcY8EXReEMzoQlSG3SAfRfW5mQ0gAi3QKPU8+Hla79oXu+yH2Ofe34DDzgX2caW0m+h7Uese5i5QUgs8ompe53S6+je2ZsHfYoiK06yiskKusRVVUfucGVXEl4wIgVBdx74M3PGDz7GCterhAGjXMO+Lqcp1yGAx6m7TDAN0Ve/2dtMNfeV/y9d4rCPCKoLlbJi/vNmmLX5MRnvAXTVovLKm3QQjLnR6dqqtCR5eHIqAlOM+h31bTG7Z5x71p3rEyKVJ2gS60XVUd16uqI6IVme00chJ1OqlK9YFKaU97rRtUyFG/t0GFIE2GMhXUR1eluVqpNDQJBwdCvWWmw63DeGweh5rLPiXbqI8Vfvla9XFrpBQbMPSTdZ8f3Q1Ix1+5ZwLffyq4cWqgd0+ha1GZ5wb6xNYAHH51K86IgSN7drVti+2Q4qTKUAZlri9fgISASer+tYiWTdUyJkgm9uOk0i0HtVKNahv/7ZubrviEDN2pKDfms++7XTIHxQzTFsVzqc+KsgnJt2GUrEjBIWjDRR5QM9dpNc47CbCYEh45v1uncl4+36RxjrzegHPCO2ic22RETHIu0wfr1Dvi+Q7hcCL5Bk2T+nzrXwwxvcySa4O8+lNyE7g9DMmQSj0VT5C2cTZl2z4rGwX4xckG+fUjGc4zI31NC0+HUKdcIoAtpPP5Gtp+rXSij6NnmgjRBV1xbggi9GrozEEqF2FQCg25CzarBo7zlz4XSAtbiXtM5oxDQeTbAB9k4gcy/WqO/dpg1gOkvEyjIqd5DWLljf5G24hOYaqrWmTHFl8hEBGhqZKOf32T7daPwga3MgaxVoReEQYCa12drROgw37vCUT/fFebfYEUJnG/rIVKXSM3DRUpQoJc2EZZyZK45r2F6YpMyRk/6C+WylOeejdJJy0CVbXy6jjWyBsnfsGVSbvZa1IGUZLRgInb/rU6m8tVk00IH9euHfAGAT4ebLLA3JYlRYmFa3UBXU3fk1XkviDCIjsPAEu8HdEZr1jUfIBtzPDzP8oMv7u8OTt79urszCOKPCytfw5R1KbOvI/lcGB3TLwKJ9YunxUTXUQ8vD4feFxIj3ie3WJkIVksCRmS2tA1tbwZIxXPubGOroLS2yjY/Jc3I8S3h4050cIoF4bxxYJ3PaIeLFl4eiUrOjOMGOlmhnjR63e3blTb9rjaRoc8ymXcFhj4hHPifTLh0Mc3md7Dm/fiGfpI1Lr81WC+JJbNal3aHY3xYrBZYfT7Xr+ePmpw+fOjp1lu9SFbBLvW89wwR56gwWL9PqMqBP+JCsJpybdmpapUZV5idieAgYPiTZIEZSRg7XqsLARuaYZk3F75ORA36nXPjBuu6d/XGnsd2DnUcrpK8pforNFqx6vbWuS6nbjXGjJABkkaquJG7jGoW1+akHiAvV8ebxqrhIczb0nsVV3reOB7tn5lKnl/U6oyhV6pC/L2jKmAqz4yW4+FaZLHxlsc6RQiRy8DB9N0BT5vwD2vscjrljdZRroRY32B9WFSQ8luC0w3MGLG8FEqXF/fYYKs/VBiax9ZJ1QVEcqtr/XJSdtSn+jAyhrp6m1a6uq2FnlaQ5Z8CRvWpModHwVAS4xGaocwm0KUae0BJXGILH9NsJmwQZHByfFakgygMlq4//CknSbH7n07s/9FCMRSIP9vLdlNmiCyxHlmbk6E105oityqkAxSiq8VcPMBhXg3A9euRqxmhNwSFgVixmKJBXqgVG0zUU9/HnEmSyuxylhZS6R744YpdnZdL7BZV6l/tbD9IlSBh13yaXONb1BgcoMzZHiDVfbP5qN1q8vXchgNfc3qC3sjIyRI82TNyrbx+nNv0Fu7sIPBJl6vbtue1wPeytCkw4Xe4HBFq00P0N3jlbS9R56kx40EBds1/dAnBcAY/aZBkqM2khy18Xr/ZBNJjjawehtJHuBkRTYLXRHlF3NcXM7NN0q2Ioa+WqXPHqcJ2QiU3tWp8YFGjgYTH/5aN3sv22jxUruh62Lim2hR3bY9LaamGrVGCF0XzZ/g2U0L6q/2cP+YstWZ2IdNYQe1TpYrzAobEn4J1eOK8By1gIJDDsGfrLcU/cFmXrH37awW38t0wp2ETb9TlJQmyawrrjjQBljP3U+N2iufctqsBy4DjfQEBM6krYzVyg3Vn3a3L+MFpwRWg3tb0sGamPQjenRMGEL8yEH/6nsCGvzbXgOmhYV51VxXN8iqtfF6TeovRyoNNIqvsci5PWFy7zsixQt6LQbFO6nTJEY/zf2sGFmTkDSZZVjE8fKBZGuFUfq94zZGeVFW1TYY5eRkg/l0bmuD6E+HZc2TjDl7x75h5bczjpwi9DaynUC5McxvRajy9SsRFNyZt6HFx2PxKlXc3vc86e4I5pCDR+jsGQq1H1W6bJAczkRSoAXm1XWN5M9ftGmufhuIO94omf2vAHEz9GdZ0eE/8VHxqkgnVbRzO3p8HN6K/TGQ65TWosl/aHwwGYcqCjri8n2NHC9O2hjwpM2ovehvYsCTr8dvtwZw2Xb8HwfMil0NyTExdDit7y5GgJYTjuZLBqb0WZQoDzlEXBQlpn/QiOADMJ7h9vUcOye1AxTg+VGyqkDeovgjUnIsziZpck4qaTdTQ3cxTi1Sbhep5nC2Gpj7QqEfA5pO0ZqtASLrUffA663n4cEGp/PYve/rVo3IfKF8bUDQB842P7NlJtw3MJClR6rbfvuJF5miaQu2uZESOiTItOwiZpO1BKc87uGnm3XrdBGSpHzz+oV8VEtUUDRX8JM+TCAyLmN3266evkd8L7jrJW+yypIi9dXKCsJokc6/bkCnVp0/8GwDyObqHW30I6v7dlf6Q3bBO8ixefzBq674WEhgwqjsEI6iufXkjeVM5sl8BYV9MMfFHZpF7kphNMCApXmGnTaB7vSezVaAqfg79z778PcafQ+PW6Wj14ZNkWzdIB29DeD0CekYAiFMSuylG4FoVkc4RVYBT1NVW6ZOdOn3oe4rmKPTUb9fxUy1zNGZH07+XGUi1y9Skvm6iKm2RNd8FFs4yz1A266PvsdXRb5GcZHIE9nZgWo4DYMXrbD4pXe4fmUOB5tWxrnvW+HOyxrcITtBgC0M2GQ8e/vlhndShjqSkK0nc8RVIjUav1dfUD+G3tqRtffbmvRztJRNfY7KKO72lfPkXVqfT2kBJsmKVT9pdc+O22Dli+ebNMzxU7BynepN4plcUbw4SHgvHivfSdluTxR74wrka6PFcavb/kK3UFmnDjbRorpte1pkCxKXaAXxDfVhQQgNkaLt3fYqYscFQg0JDPM8GxGKnK7GMAYtxCAd1yKB/U2M4dy2PTH8ZJI9rglinpvjAn/8IZRoVlA6hOi3EcLkOtYQYlPY3rlte0LwZvccG1JT/YW9eryPzyKciLO1dNM2lHlPkMubh5Hiz6Q8rMjLq5RsSrSMVw1ovz2j0eIhDY42ZzR295BM9N8bRcUK/H5Fx26UnIlPxCxxSbltdYjJ+n8vHs3t4uLuvJHtcb+M7PiOg/YI4GA9ZY6eiAAOdqZMqmZyVXZu+SjgFjH8rjo1XST6627O3WWkK5TzsCPe1JVqO4Y91t9jXhO3GGwy5NVtO0Osc1NRah4Exx9NqRAhQtAIEMWzxetOeZb1FAOzD804IvobXqb5n/6eTqBygsF0PtLtovjbR+XnvcsoFDeVRASL/ZUH3kiyTuhRsGEqu+rybk+It4prnXdM49i7EBOx2YFSEcr5KA2DiTKVJ64ibF3Kwzb7ONic1jp8ykBu4Y6QX60//MSJzDIKxZVBnpvSLxs7rqf2Mlkm62KAfzfHkSFaFFFWpgG29uFVmrBVL1NBpZtjif4eyP8V8kqrgb+XbXEXuOL9lvzK0RMufP8PcuG74vLujL9S6eNzDqbiC4KAr8FDWOz2eLe38I/CdnnQnxPTFe5oPVl2i/55yJ+nMC3XWkIvdo96c8nKzes2bCv2sb39YDdt9/r2hjszIva9ZsEC+C0//1RH08/bQ4ZtaLq3OWT49UFad0f/j+Ltxc2tuLg5c/OagdMnZ10GSYXBSlb9FxwUr0uO31H1XLC21FvgsQ94hbLt3vpJm7fenlQ/2phB3phVf9KUmErHSI1NmiMlfYzn2iKIVPE3RU3QqmrsrDswZ36Ket/qY/a8bcbx2Ll6mM9ylyf9CXv+WJav3C/XTHSjUEmSZvfK6m/KcYK6LXOc+mpN1piOinfyER82IYX3KVyojCD2ljDyHLKCYsxJVVNjV5XjoIFObDSCXCftfkZbrnSw2c/YPVdK2DGfrgGP+juaC8IGyzV02Kw+bitXnNQhoaVkMS+rVBxuZ4ywXve/aMXXA2+wHjsdn2wO/w12dkgfuCRzRQ3wUVGP7+ymCoZAWoh2oM1ZGVl6Ug8MXrbFhrD066Oih0dPRO16R39IbKjf07GhtVInR/jw1iRpkvLMHK/vDdqB07gvfpulukvwnbaE3NoPjYTioL1uo9/fNaH43L1tdyqeyxgf4YzZQmHDsA5ZonAvR5i5NWHILaCbFL2MCeGi7gt7dXYk5qcwCvizGd8jTDDjr6fU9gyUQJl8Byluie/F/s9DF9uJh1AK9oYO6k7hUXtVWEtI5fiJqrDdQyo5yphWS2Xu9GHvLJsukiQS77Rt/gbrPqb5E4DT351ppBnpyXKBrA8olywyptjwTZ1cg+etQeB+G71IOWyGU7sTjOyt3QBSkev1h/Prq/ITkDviyqqzzPpaq6GSeQ5T+aluDl4ONkSiXu4eq3Vu254cEzWbraiwtzgozuUizHe1AoSV0KZTIGieL7fFgoNeezVE78VXVBM69321U2SrIRi0/dA/dtMtwqR37O5kj3cn13Eh38et36BETK+V9WovDVbi5bcXZyZJWdvqtQNjmruriEMZJp4mKg6/dMTZP+r45Hl78qD3NWWuxxuSy63YrVjGEQqXVvBbsfzAFU0EqXP9AeUd+NK9pwyE4qtT8QO6mxAtbuss+fy4PdXeUgd59Ly32U8f9P6oVLtbbrWXoYcSZ4t0ftGk4MuvrnYJ7HqIJetgF/Ml5+G74u4xcerAiDfr7SV8meXP7O63ZlMJ52u4LR+/3fLTt8igyUCVHwYrezbUN/8k/PmoT9Mw54qW1d0+5pL1X/2tffL3/PU1AQKiyjhxKk3r37GFLnX4091poMYSTYLRRGwWrrZE4PPbjOLuG0YBllDibJZYT8fd/IhzWwyAfFSnPc2616Nj/NrXlx8pWt0nTWe6vFd6qwGcly209cel1o5i0DKKVygW+aS+RGt2q42ANR/53HYj4c0SOxGh+UUM5+0Bn+qmMtni5U9+c3jtqHoto7qM4ebL9EFG09WBobCo65uzO46NnGc0qN5piRp93mutQ+hMdxbOtxiF3fFre9GvHcKh1V/2c932y3fmu2X39oPN6xTMamNUZ6jcXaeb8cktRjtU/EWx8+GV2K/goY1Xlh/QtvGu58hZGBvRQkVdNnWlpvnq4CSr8jnObaN0MChNuLIAAp+YlzPTxcPaFJQKH7QMh9wJ8gXuUvuNXWc0U5zqku8U70ioBX9zvda3tO31CM4VGeEAMt3L1R4vfLab8tntB3FmKnVe242oYj/j7VN8mNYQuob7N5bb/9uoo/dBnU/VGvLwua5P57YZmkV+pVpoeeUd3Vv0eqqfuc1wnPdip2lmOt9swSSVUua2nuYzfWtf/QGevriOsixZnS6HAboJn9zivbfojVkodzPylf1Q+3oNvNrpz1XCONuV+uwWrzf6zb7ToqxLpz1iQ8XY/mT3zvceN6qZZqtC12rjVDdcbmWvhrkcV8ZC7GuwB7xvFEm77Kx08q3JDum5R3N2e9kZfnptu2WUX3UU+7bq3/1s5HruXeno6XIuTnbN1vOdRrTPKajvTcPVtpc3v9vjalac6o7o1DYvpjdeXop9t7N+x3aFbHv5mg+jOO9fKg2m+OyOc+dmxPzJ1LZ3N1oEN+ETznYTfXaH6eMjqx1x8bdW/nO7kbucp0JuI7bFm25fN6RQBr9K9Hh3pbBNAN1WlKsKg092pzi5xUhKXu80+3C2TJ5sa+aH6EBxFpAqnofrkIa+oiv1FbsN45PZ0ytIZvQ3mNeO4y2sJxT86rpPYDr51BZvfoNgtCKnz/mWq+tC/v5/AXE+j9EmngAA"
_files = json.loads(gzip.decompress(base64.b64decode(_BLOB)).decode())
for relpath, content in _files.items():
    full = os.path.join(os.getcwd(), relpath)
    os.makedirs(os.path.dirname(full), exist_ok=True)
    if not os.path.exists(full) or os.path.getsize(full) == 0:
        open(full, 'w').write(content)
        print('wrote', relpath)
    else:
        print('ok   ', relpath)


## 3. Configuration

In [ ]:
# === EDIT THESE (defaults are A100 40GB tuned) ===
PERSONA = 'priya'         # or 'rohan'
MODEL_SIZE = '7b'         # A100/L4: '7b'  |  T4 free: '3b'
EPOCHS = 3
MAX_SEQ = 4096            # A100 can hold 4096 for 7B 4-bit; drop to 2048 on L4/T4
BATCH_SIZE = 4            # A100: 4  |  L4: 2  |  T4: 1-2
GRAD_ACCUM = 4            # effective batch = BATCH_SIZE * GRAD_ACCUM (=16 on A100)
# =================================================

MODEL_MAP = {
    '1.5b': 'unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit',
    '3b':   'unsloth/Qwen2.5-3B-Instruct-bnb-4bit',
    '7b':   'unsloth/Qwen2.5-7B-Instruct-bnb-4bit',
}
BASE_MODEL = MODEL_MAP[MODEL_SIZE]
BASE_MODEL_FOR_INFERENCE = BASE_MODEL.replace('unsloth/','Qwen/').replace('-bnb-4bit','')
print(f'Persona={PERSONA}  Model={BASE_MODEL}  Epochs={EPOCHS}  MaxSeq={MAX_SEQ}  EffBatch={BATCH_SIZE*GRAD_ACCUM}')

## 4. Install dependencies

Takes ~5 min the first time.

In [ ]:
%%capture
!pip install -U pip wheel
# unsloth pulls compatible torch/transformers/peft/trl/bitsandbytes
!pip install 'unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git'
!pip install --no-deps trl peft accelerate bitsandbytes
!pip install sentence-transformers faiss-cpu datasets

In [ ]:
# Smoke test
import torch, transformers, peft, trl, faiss, sentence_transformers
print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

## 5. Prepare training data

In [ ]:
# Remove older generated files for this persona, then regenerate
!rm -f training/data/sft_train_$PERSONA.jsonl \
       training/data/sft_eval_$PERSONA.jsonl \
       training/data/rag_docs_$PERSONA.jsonl \
       training/data/rag_meta_$PERSONA.jsonl \
       training/data/rag_index_$PERSONA.faiss \
       training/data/system_prompt_$PERSONA.txt
!python3 training/prep_training_data.py --persona $PERSONA

## 6. Build the RAG index (embed + FAISS)

In [ ]:
!python3 training/build_rag_index.py --persona $PERSONA --batch-size 256

## 7. Train the LoRA

**A100 40GB**: ~25–40 min for 7B at MAX_SEQ=4096, effective batch 16.
**L4 24GB**: ~90 min for 7B at MAX_SEQ=2048.
**T4 free**: ~60–90 min for 3B.

Checkpoints save to your Drive every 200 steps, so a session disconnect won't lose work.

In [ ]:
# Inline trainer (Colab-tuned — saves to Drive, configurable model size)
import os, json
from pathlib import Path
from unsloth import FastLanguageModel
import torch
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

DATA_DIR = Path('training/data')
CKPT_DIR = Path(f'training/checkpoints/{PERSONA}')
CKPT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Loading {BASE_MODEL}...')
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL, max_seq_length=MAX_SEQ, dtype=None, load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=32, lora_dropout=0.05, bias='none',
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    use_gradient_checkpointing='unsloth', random_state=42,
)

def format_example(ex):
    return {'text': tokenizer.apply_chat_template(ex['messages'], tokenize=False, add_generation_prompt=False)}

train_ds = load_dataset('json', data_files=str(DATA_DIR / f'sft_train_{PERSONA}.jsonl'), split='train')
eval_ds  = load_dataset('json', data_files=str(DATA_DIR / f'sft_eval_{PERSONA}.jsonl'),  split='train')
train_ds = train_ds.map(format_example, remove_columns=train_ds.column_names)
eval_ds  = eval_ds.map(format_example,  remove_columns=eval_ds.column_names)
print(f'Train: {len(train_ds)}  Eval: {len(eval_ds)}')

config = SFTConfig(
    output_dir=str(CKPT_DIR), num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE, gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=2e-4, lr_scheduler_type='cosine', warmup_ratio=0.03,
    bf16=torch.cuda.is_bf16_supported(), fp16=not torch.cuda.is_bf16_supported(),
    logging_steps=10, eval_strategy='steps', eval_steps=200,
    save_strategy='steps', save_steps=200, save_total_limit=2,
    max_seq_length=MAX_SEQ, dataset_text_field='text', packing=False,
    report_to='none', seed=42,
)

trainer = SFTTrainer(model=model, tokenizer=tokenizer,
                     train_dataset=train_ds, eval_dataset=eval_ds, args=config)
print('Starting training...')
trainer.train()
trainer.save_model(str(CKPT_DIR))
tokenizer.save_pretrained(str(CKPT_DIR))
print(f'Saved adapter to {CKPT_DIR}')

## 8. Quick inference test

Loads the LoRA you just trained + the RAG index, asks one question.

In [ ]:
# Reuse the trained model still in memory — no need to reload
import faiss, json, re, torch
from sentence_transformers import SentenceTransformer

DATA_DIR = Path('training/data')
embedder = SentenceTransformer('BAAI/bge-base-en-v1.5')
rag_index = faiss.read_index(str(DATA_DIR / f'rag_index_{PERSONA}.faiss'))
rag_meta  = [json.loads(l) for l in (DATA_DIR / f'rag_meta_{PERSONA}.jsonl').open()]
sysprompt = (DATA_DIR / f'system_prompt_{PERSONA}.txt').read_text()
FastLanguageModel.for_inference(model)

def retrieve(q, k=8):
    e = embedder.encode([q], normalize_embeddings=True).astype('float32')
    s, idxs = rag_index.search(e, k)
    return [(rag_meta[i], float(sc)) for i, sc in zip(idxs[0], s[0]) if i >= 0]

def ask(q, k=8, max_new=600):
    retrieved = retrieve(q, k)
    sources = '\n\n'.join(
        f"[Source {i+1}] {c['meta'].get('title') or c['meta'].get('doc_id')} ({c['meta'].get('date','')})\n{c['text']}"
        for i, (c,_) in enumerate(retrieved)
    )
    user = f"Answer the question using ONLY the source documents. Cite [Source N] inline.\n\nQUESTION: {q}\n\nSOURCES:\n{sources}"
    msgs = [{'role':'system','content':sysprompt},{'role':'user','content':user}]
    text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new, do_sample=True, temperature=0.4, top_p=0.9, pad_token_id=tokenizer.eos_token_id)
    answer = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
    print('ANSWER:\n' + '='*70)
    print(answer)
    print('\nSOURCES:')
    for i,(c,sc) in enumerate(retrieved):
        print(f"  [{i+1}] {c['meta'].get('doc_id'):40s} {c['meta'].get('date','')} score={sc:.3f}")
    return answer

# Try one
demo_q = {'priya':'What happened with Acme Corp?','rohan':'Why are we using Postgres for events instead of Mongo?'}[PERSONA]
_ = ask(demo_q)

## 9. Interactive Q&A

Run this cell, then keep clicking it to ask new questions, OR paste calls to `ask('...')` in new cells.

In [ ]:
# Try the demo questions for your persona
PRIYA_DEMOS = [
    'What happened with Acme Corp?',
    'Why did we credit Acme $4,200?',
    "What's the soft commit on Acme's Premium upgrade?",
    'Why did Rekall churn?',
    'Who is Mike Reyes and how should I handle him?',
    'Tell me about the Umbrella security questionnaire.',
    'What was the May 2025 Hooli incident from the customer side?',
]
ROHAN_DEMOS = [
    'Why are we using Postgres for events instead of MongoDB?',
    'What was the May 2024 incident?',
    'Why did we kill the GraphQL gateway?',
    "What's the rate-limiter v2.1 design recommendation?",
    "What's the trigger to revisit ClickHouse?",
    'What does ADR-0023 say?',
    'Who should own ratelimit-v2.1?',
]

# Pick one or write your own
_ = ask('What happened with Acme Corp?')

## 10. Run formal eval

In [ ]:
# eval.py reloads the model from disk; that's fine. Could take 10-15 min.
!python3 training/eval.py --persona $PERSONA --base-model $BASE_MODEL_FOR_INFERENCE

## 11. Train the OTHER persona

Restart runtime (Runtime → Restart), change `PERSONA` in cell 3, run all again. Both adapters end up in your Drive.

## 12. Verify everything is on Drive

Because the project root IS your Drive folder, training output is already saved. Confirm:

In [ ]:
!du -sh training/checkpoints/$PERSONA training/data/rag_index_$PERSONA.faiss training/data/rag_meta_$PERSONA.jsonl
!ls training/checkpoints/$PERSONA